# 2026-10-20 Run durations, outbound and return

**Author:** Kyle Shannon · **Notebook:** run-durations-outbound-vs-return · **Created:** 2026-10-20T13:58:53-07:00

**Purpose:** How long the lab's labelled runs last on the track, the outbound runs (paths 1 to 4) against the return runs (paths 9 and 10), as two distributions over every transformed session. The generator draws run durations per path from these, so this is the real side of that fit.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401  registers the SciencePlots styles

import axis_of_travel_rnn
from environments import list_environments, load_environment
from axis_of_travel_rnn.store import resolve_store

plt.style.use(["science", "notebook", "no-latex"])
rng = np.random.default_rng(0)
store = resolve_store(None)
FIGURES = Path("figures")

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"axis-of-travel-rnn {axis_of_travel_rnn.__version__}  commit {commit}  {date.today()}")
print(f"store: {store.root} ({store.source})")
print(f"environments: {', '.join(list_environments())}")

## Data

The converter's `runs.csv` and `epochs.csv` from the `TRANSFORMED_LAB_DATA` folder named in `configs/local.yaml`, read only. A run is one of the lab's labelled runs with its `path_id`, `duration_s` and `clean` flag. Outbound runs are paths 1 to 4 (11 to 14 in a rotated second segment), return runs are paths 9 and 10 (19 and 20). Only track epochs are kept. Nothing is copied into the repository, the figure holds aggregates only.

In [ ]:
from proj_helpers.paths import local_config, repo_root

OUTBOUND = {1, 2, 3, 4, 11, 12, 13, 14}
RETURN = {9, 10, 19, 20}
IMGS = repo_root() / "figures"

root = Path(local_config()["TRANSFORMED_LAB_DATA"])
runs = pd.read_csv(root / "runs.csv")
epochs = pd.read_csv(root / "epochs.csv")
track = runs[runs["epoch"] == "track"].copy()
track["kind"] = np.where(track["path_id"].isin(OUTBOUND), "outbound", np.where(track["path_id"].isin(RETURN), "return", "other"))
print(f"transformed data: {root}")
print(f"{runs['session_id'].nunique()} sessions, {len(track)} track runs, by kind:")
print(track.groupby(["kind", "clean"]).size().unstack(fill_value=0))
print(track.groupby("kind")["duration_s"].describe().round(2))

## Analysis

Clean runs only in the main panel, since a dirty run (the labeller's own flag, the rat stopped or turned) is longer by construction and would smear the right tail. The dirty runs are counted in the caption. Both kinds share the bins so the shapes compare, and each is drawn as a density so the different counts do not set the heights.

In [ ]:
COLOURS = {"outbound": "#2a78d6", "return": "#eb6834"}
clean = track[track["clean"] & track["kind"].isin(COLOURS)]
dirty = track[~track["clean"] & track["kind"].isin(COLOURS)]
upper = np.ceil(clean["duration_s"].quantile(0.995))
bins = np.arange(0, upper + 0.25, 0.25)

fig, ax = plt.subplots(figsize=(9, 4.5))
for kind, colour in COLOURS.items():
    values = clean.loc[clean["kind"] == kind, "duration_s"]
    ax.hist(values, bins=bins, density=True, histtype="stepfilled", color=colour, alpha=0.25, lw=0)
    ax.hist(values, bins=bins, density=True, histtype="step", color=colour, lw=1.5, label=f"{kind}, {len(values):,} clean runs")
    median = values.median()
    ax.axvline(median, color=colour, lw=1, ls="--")
    ax.text(median, ax.get_ylim()[1] * 0.98, f" median {median:.2f} s", color=colour, va="top", ha="left", fontsize=9)
ax.set_xlim(0, upper)
ax.set_xlabel("run duration (s)")
ax.set_ylabel("density of runs")
ax.set_title("Run durations on the track: outbound against return")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)

n_sessions = clean["session_id"].nunique()
caption = (
    f"Read: each curve is the share of that kind's clean runs per 0.25 s bin, dashed lines the medians. "
    f"Outbound is start to goal (paths 1 to 4), return is goal to start along the perimeter (paths 9 and 10).\n"
    f"Data: {n_sessions} sessions in {root.name}, {len(clean):,} clean track runs, {len(dirty):,} dirty runs left out "
    f"({(dirty['kind'] == 'outbound').sum():,} outbound, {(dirty['kind'] == 'return').sum():,} return). "
    f"Made {date.today().isoformat()} at commit {commit}."
)
fig.text(0.01, -0.08, caption, ha="left", va="top", fontsize=8, wrap=True)
fig.tight_layout()
IMGS.mkdir(exist_ok=True)
fig.savefig(IMGS / "run_durations_outbound_vs_return.png", dpi=300, bbox_inches="tight")
print(f"saved {IMGS / 'run_durations_outbound_vs_return.png'}")

### Per path

The same durations split by path id, since the generator draws per path and the four goals sit at different distances.

In [ ]:
paths = sorted(clean["path_id"].unique())
fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(len(paths))
groups = [clean.loc[clean["path_id"] == p, "duration_s"].to_numpy() for p in paths]
colours = [COLOURS["outbound"] if p in OUTBOUND else COLOURS["return"] for p in paths]
parts = ax.boxplot(groups, positions=positions, widths=0.6, showfliers=False, patch_artist=True, medianprops={"color": "black"})
for patch, colour in zip(parts["boxes"], colours):
    patch.set(facecolor=colour, alpha=0.35, edgecolor=colour)
ax.set_xticks(positions, [f"path {p}\n{len(g):,}" for p, g in zip(paths, groups)])
ax.set_ylabel("run duration (s)")
ax.set_title("Clean run durations per path, outbound in blue and return in orange")
ax.grid(axis="y", alpha=0.3)
fig.text(0.01, -0.06, f"Read: box is the middle half of the runs on that path, line the median, whiskers 1.5 times the box, outliers hidden, "
         f"the count under each path. Data: as above, {len(clean):,} clean track runs from {n_sessions} sessions, made {date.today().isoformat()} at commit {commit}.",
         ha="left", va="top", fontsize=8, wrap=True)
fig.tight_layout()
fig.savefig(IMGS / "run_durations_per_path.png", dpi=300, bbox_inches="tight")

In [ ]:
shortest = track.loc[track.groupby("path_id")["duration_s"].idxmin(), ["path_id", "session_id", "run", "duration_s", "clean"]]
print(shortest.sort_values("path_id").to_string(index=False))


## Figures

`imgs/run_durations_outbound_vs_return.png` and `imgs/run_durations_per_path.png`, both at 300 dpi.

## Notes

_what the distributions say about the generator's per path draw, and anything odd in the tails_